# Day 24: LLM APIs and Efficient Prompting

On Day 23 we looked **inside** an LLM. Today we **talk to a real one** from Python.

## 🎯 The example we will build

A **tea shop** gets customer messages. We build a small program that reads each message and sorts it:

| Message from the customer | What our program returns |
|---|---|
| *The chai was cold and the delivery was late.* | `{"label": "complaint", "reason": "cold chai, late"}` |
| *Best paratha in town!* | `{"label": "praise", "reason": "likes the paratha"}` |
| *Do you deliver to Gulberg?* | `{"label": "question", "reason": "asks about delivery"}` |

- The **LLM** does the reading. We only send the message and read the answer.
- The answer comes back as **JSON**, so our Python code can use it.
- **Home Problem:** the same steps on a new job. Read order messages and return the customer's **name, city and items**.

**Which model?** Only the **free NVIDIA Nemotron models on OpenRouter**:
- a **chat** model, for the sorting
- an **embedding** model, for Step 5

## 📋 Agenda

We climb from the bottom. Each step fixes a problem of the step below.

```
TOP     Step 10  Put it together          the full tea-shop program
          ▲
        Step 9   JSON output              code can use the answer
          ▲
        Step 8   Second message           keep the context, stay under the limit
          ▲
        Step 7   System prompt            short rules, no extra talk
          ▲
        Step 6   Zero-shot, few-shot      show examples
          ▲
        Step 5   Embeddings               text as numbers (Day 23, for real)
          ▲
        Step 4   Tokens = cost            what we pay
          ▲
        Step 3   Read the response        find the answer inside
          ▲
        Step 2   Send a request           ask the LLM
          ▲
BOTTOM  Step 1   Connect                  API key
```

**What is an API?** *Everyday example:* a restaurant.

| Restaurant | Python and LLM |
|---|---|
| You (the customer) | Your Python code |
| The waiter | The **API** (carries your order and brings the food) |
| The kitchen | The LLM (far away, on someone else's computer) |
| Your order | The **request** |
| The food | The **response** |
| The bill | **Tokens** = money |

### New words today

| Word | Meaning |
|---|---|
| **API** | A way for one program to ask another program for something. |
| **API key** | A secret password that says "this request is from me". |
| **OpenRouter** | A website that gives one key and one address for many LLMs. |
| **Request / Response** | What we send / what we get back. |
| **Model** | Which LLM to use. A name ending in `:free` is a free one. |
| **Message and role** | One piece of the chat. Roles: `system`, `user`, `assistant`. |
| **Prompt** | The text we give the model. |
| **Embedding** | A list of numbers that describes a text's meaning (Day 23). |
| **Zero-shot / Few-shot** | Ask with **no examples** / ask with **a few examples**. |
| **Temperature** | `0` = always the top choice. Higher = more random. |
| **JSON** | Text in a fixed shape that code can read. |

---
# Class Problem: sort the tea shop messages

## Before we start: get your free key

1. Go to **openrouter.ai** and make an account.
2. Open the **Keys** page and create a key. Copy it.
3. Do **not** paste it into the notebook. The next cell asks for it and hides it.

**Good to know:**
- Free models are slow sometimes. If you see an error with **429** or "rate limit", wait one minute and run the cell again.
- Model names change. If a name gives an error, open **openrouter.ai/models**, filter for free, and copy a new name.

In [24]:
import os, json                         # os: read secrets, json: read JSON text
import numpy as np                      # numbers and vectors (Day 23)
from getpass import getpass             # asks for a password without showing it
from openai import OpenAI               # the "OpenAI-style" client. OpenRouter copies this style.

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"       # embedding model (free, NVIDIA)

key = os.environ.get("OPENROUTER_API_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)     # the web address of OpenRouter

## Step 1: Connect (authentication)

- The **API key** is a password. OpenRouter uses it to know **who is asking**.
- That is why the code above uses `getpass` (the key is hidden).

**Rules for keys:**
1. **Never** write the key inside the notebook or the code.
2. **Never** push it to GitHub. Robots search GitHub for keys.
3. If a key leaks, delete it on the website and make a new one.

**One more rule:** free models may **save what you send**. Never send private things (phone numbers, passwords).

**✏️ Quick Check:** Why do we use `getpass(...)` and not `api_key = "sk-or-abc123"`?

**Answer:** With `getpass`, the key is never saved in the file. If we share the notebook or push it to GitHub, nobody sees our key.

## Step 2: Send a request

A request needs two things:

- **model**: which LLM to use.
- **messages**: the chat so far. It is a **list**. Each message is a small dictionary with a **role** and the **content** (text).

Today the list has one message from the **user**.

In [25]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello to Sara in one short sentence."}],
)
print(type(response))        # a big object, not just text

<class 'openai.types.chat.chat_completion.ChatCompletion'>


## Step 3: Read the response

The answer is **inside** the response object. We dig down to it:

- `response.choices[0].message.content`: the **text**.
- `response.usage`: how many **tokens** we used (Step 4).
- `response.choices[0].finish_reason`: why it stopped. `"stop"` = finished normally. `"length"` = it ran out of space.

In [26]:
print(response.choices[0].message.content)        # the answer text
print("stopped because:", response.choices[0].finish_reason)
print(response.usage)                             # prompt_tokens, completion_tokens, total_tokens

Hello, Sara!
stopped because: stop
CompletionUsage(completion_tokens=142, prompt_tokens=25, total_tokens=167, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=136, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0})


**✏️ Quick Check:** Print **only** the number of `total_tokens`.

In [27]:
# your code here

**Answer:**

In [28]:
print(response.usage.total_tokens)

167


## Step 4: Tokens = cost

Remember tokens from Day 23? Paid models charge for **tokens**, not words.

- **Input tokens** (`prompt_tokens`): what we send.
- **Output tokens** (`completion_tokens`): what the model writes. Usually **more expensive**.
- Prices are written **per 1 million tokens**.

**Cost = (input tokens × input price + output tokens × output price) ÷ 1,000,000**

Our models are **free**, so the real cost is $0. We still learn the formula, because every paid model uses it.

In [ ]:
PRICE_IN, PRICE_OUT = 0.15, 0.60        # dollars per 1 million tokens. EXAMPLE prices of a paid model

def cost_of(usage):                     # one function, used again and again
    return (usage.prompt_tokens * PRICE_IN + usage.completion_tokens * PRICE_OUT) / 1_000_000

print(f"This request would cost ${cost_of(response.usage):.6f} on a paid model")

This request would cost $0.000089 on a paid model


**✏️ Quick Check:** A request uses **200** input tokens and **100** output tokens. What does **10,000** requests cost?

In [30]:
# your code here

**Answer:** one request = $0.00009, so 10,000 requests = **$0.90**.

In [31]:
one = (200 * PRICE_IN + 100 * PRICE_OUT) / 1_000_000
print(f"one request: ${one:.5f}")
print(f"10,000 requests: ${one * 10_000:.2f}")      # $0.90

one request: $0.00009
10,000 requests: $0.90


**Efficient prompting starts here:** shorter prompts and shorter answers = lower cost and faster replies.

## Step 5: Embeddings from a real model

On Day 23 we **wrote** 4 numbers for each word. A real embedding model gives **thousands** of numbers, learned in training. It does not chat. It only turns text into numbers.

| New word | Meaning |
|---|---|
| **Embedding model** | A model that turns text into an embedding. |
| **Dimension** | How many numbers are in one embedding. |
| **Cosine similarity** | Like the dot product (Day 23), scaled so the answer is between -1 and 1. **Close to 1 = similar meaning.** |

We send 4 words. We get one embedding for each.

In [32]:
words = ["tea", "coffee", "apple", "laptop"]

result = client.embeddings.create(model=EMB_MODEL, input=words)       # one request, 4 words
vecs = np.array([item.embedding for item in result.data])             # one row per word, like E in Day 23

print(vecs.shape)          # (4, N): 4 words, N numbers each. In Day 23: (6, 4)

(4, 2048)


In [33]:
def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))     # cosine similarity

for i in range(1, len(words)):
    print(f"tea vs {words[i]:7} = {similarity(vecs[0], vecs[i]):.2f}")
# coffee should be the highest: both are drinks. laptop should be the lowest.

tea vs coffee  = 0.50
tea vs apple   = 0.25
tea vs laptop  = 0.17


Nobody told the model that tea and coffee are both drinks. NVIDIA trained it on huge text, so it **learned** that they appear in similar sentences.

**✏️ Quick Check:** Which is most similar to **"samosa"**: pakora, paratha, laptop or tea?

In [34]:
# your code here

**Answer:**

In [35]:
more = ["samosa", "pakora", "paratha", "laptop", "tea"]
v = np.array([item.embedding for item in client.embeddings.create(model=EMB_MODEL, input=more).data])
for i in range(1, len(more)):
    print(f"samosa vs {more[i]:8} = {similarity(v[0], v[i]):.2f}")
# pakora is usually the highest: also a fried snack

samosa vs pakora   = 0.26
samosa vs paratha  = 0.21
samosa vs laptop   = 0.13
samosa vs tea      = 0.18


## Step 6: Zero-shot, then few-shot

Now our tea shop job. First, a small helper so we do not repeat code:

In [36]:
def ask(messages, temperature=0):                     # temperature 0 = always the top token, like argmax in Day 23
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                            # no key? rate limit? wrong model name?
        return f"ERROR: {e}"

message = "The chai was cold and the delivery was late."

### Zero-shot = no examples

We only give the task. The model must guess what we want.

In [37]:
zero = ask([{
    "role": "user",
    "content": f"Is this message a complaint, praise, or question?\n\n{message}"
    }])
print(zero)      # often a long sentence, not one word. Your output may differ a little.

This is a **complaint**. The message expresses dissatisfaction with both the product (cold chai) and the service (late delivery), with no question or positive intent.


### Few-shot = a few examples

We **show** the model 3 examples first. It copies the **pattern** and the **format**.

*Everyday example:* you do not explain a job to a new worker for an hour. You show him 3 finished ones.

In [38]:
few = ask([{"role": "user", "content": "Message: Thank you, the samosas were great!\nLabel: praise\n\n"
                                       "Message: Do you open on Sunday?\nLabel: question\n\n"
                                       "Message: My order never came.\nLabel: complaint\n\n"
                                       f"Message: {message}\nLabel:"}])
print(few)       # now it should be one word: complaint

complaint


## Step 7: System prompt

- A **system** message is the **rules for the model**, written once at the top. The user message is the question.
- *Everyday example:* the rules on the shop wall, and then the customer's order.
- Good rules are **short and clear**: who it is, what to do, what **not** to do.

In [39]:
SYSTEM = "You sort customer messages. Reply with ONE word only: complaint, praise or question. No other text."

out = ask([{"role": "system", "content": SYSTEM},
           {"role": "user", "content": message}])
print(out)       # complaint

complaint


## Step 8: A second message, and the context limit

**The API has no memory.** Every request is a new conversation. To continue a chat, we send the **whole list again**, with the new message added at the end.

*Everyday example:* a friend who forgets everything when you stop talking. To continue, you remind him first: "Earlier I told you I'm Sara and I ordered 2 chai. Now, what did I order?"

| New word | Meaning |
|---|---|
| **Turn** | One message in the chat: one person speaks. |
| **Context** | Everything the model can see in this request: all the messages in our list. |
| **Context limit** | The most tokens the model can read in one request. The prompt and the answer share it. |

In [40]:
messages = [{"role": "system", "content": "You help a tea shop. Keep answers short."},
            {"role": "user", "content": "My name is Sara and I ordered 2 chai."}]

reply = ask(messages)                                               # 1. first answer
messages.append({"role": "assistant", "content": reply})            # 2. save the model's answer in the list

messages.append({"role": "user", "content": "What did I order?"})   # 3. add the second message
print(ask(messages))                                                # 4. send the WHOLE list again: it knows "2 chai"

You ordered 2 chai.


**✏️ Quick Check:** Send only the question **"What did I order?"** in a new list (no earlier messages). What does the model say? Why?

In [41]:
# your code here

**Answer:** the model has no idea (it may guess). The old messages are not in the new list, so it cannot see them.

In [42]:
print(ask([{"role": "user", "content": "What did I order?"}]))     # no memory: it does not know

I don't have access to your personal accounts, email, or order history, so I can't see what you've purchased. 

To find out what you ordered, you can:
- Check your email for order confirmations
- Log into the retailer's website or app
- Look at your bank or credit card statements for recent charges
- Provide an order number if you have one

If you share the retailer or an order ID, I'd be happy to help you track it or look up shipping/delivery details!


### The context limit

- Each time we add a turn, the list gets **longer**, so we send **more tokens** (Step 4: more cost).
- Every model has a **context limit**. If we go over it, we get an **error**, or the answer is **cut off** (`finish_reason: "length"`).
- *Everyday example:* a whiteboard. When it is full, you erase old notes or write a short summary.

**How do we find the limit?** OpenRouter has a public model list (no key needed):

In [43]:
import requests

models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]     # public list of models
info = next((m for m in models if m["id"] == MODEL), None)

if info:
    print("context limit:", info["context_length"], "tokens")                                  # prompt + answer together
    print("longest answer:", info["top_provider"]["max_completion_tokens"], "tokens")
else:
    print("This model name is not in the list. Check the name on openrouter.ai/models")

context limit: 1000000 tokens
longest answer: 65536 tokens


**How do we stay under the limit?**

1. **Keep prompts short.**
2. **Cut old turns.** Keep the system message and only the last few turns.
3. **Summarize.** Ask the model to turn 20 old messages into 3 lines, and keep only that.
4. **Start a new chat** for a new topic.

In [44]:
messages = [messages[0]] + messages[-6:]       # keep the system message + the last 6 messages
print(len(messages), "messages kept")

5 messages kept


**✏️ Quick Check:** In `[messages[0]] + messages[-6:]`, why do we keep `messages[0]`?

**Answer:** it is the **system message** (the rules). If we cut it, the model forgets how it must behave.

## Step 9: JSON output

A sentence is good for people. **Code needs a fixed shape.** We ask for **JSON**, then turn it into a Python dictionary with `json.loads`.

Models sometimes add extra things (like ` ```json ` fences). So we **clean the text first**, and we use `try/except` in case it still fails.

In [45]:
def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()   # remove fences
    return json.loads(text)                                                                      # text → dictionary

SYSTEM_JSON = ('You sort customer messages. Reply with JSON only, in this shape: '
               '{"label": "complaint or praise or question", "reason": "max 8 words"}')

raw = ask([{"role": "system", "content": SYSTEM_JSON},
           {"role": "user", "content": message}])
print(raw)

try:
    result = parse_json(raw)
    print(result["label"], "|", result["reason"])      # now Python can use the answer
except json.JSONDecodeError:
    print("The model did not return valid JSON. Run again, or make the rule clearer.")

{"label": "complaint", "reason": "Cold tea and late delivery"}
complaint | Cold tea and late delivery


**✏️ Quick Check:** Test `parse_json` **without the API**: what does it return for the text below?

`'```json\n{"label": "praise"}\n```'`

In [46]:
# your code here

**Answer:** the fences are removed and we get a dictionary.

In [47]:
print(parse_json('```json\n{"label": "praise"}\n```'))     # {'label': 'praise'}

{'label': 'praise'}


## Step 10: Put it together

One function: **system rule + few-shot examples + JSON + parse**. Then we run it on all the tea shop messages.

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [48]:
import os, json
import numpy as np
from getpass import getpass
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"
EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"
key = os.environ.get("OPENROUTER_API_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

SYSTEM = ('You sort customer messages. Reply with JSON only, in this shape: '
          '{"label": "complaint or praise or question", "reason": "max 8 words"}')

EXAMPLES = [("Thank you, the samosas were great!", '{"label": "praise", "reason": "likes the samosas"}'),
            ("Do you open on Sunday?",             '{"label": "question", "reason": "asks about opening day"}'),
            ("My order never came.",               '{"label": "complaint", "reason": "order not delivered"}')]


def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)


def classify(message):
    messages = [{"role": "system", "content": SYSTEM}]
    for text, answer in EXAMPLES:                                  # few-shot: examples as past chat turns
        messages += [{"role": "user", "content": text}, {"role": "assistant", "content": answer}]
    messages.append({"role": "user", "content": message})          # the real question
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    try:
        return parse_json(r.choices[0].message.content or "")
    except json.JSONDecodeError:
        return {"label": "unknown", "reason": "bad JSON"}


for m in ["The chai was cold and the delivery was late.", "Best paratha in town!", "Do you deliver to Gulberg?"]:
    result = classify(m)
    print(f"{result['label']:10} | {result['reason']:28} | {m}")

# embeddings: tea should be closer to coffee than to laptop
v = np.array([i.embedding for i in client.embeddings.create(model=EMB_MODEL, input=["tea", "coffee", "laptop"]).data])
sim = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"tea~coffee {sim(v[0], v[1]):.2f}   tea~laptop {sim(v[0], v[2]):.2f}")

AuthenticationError: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}

### Key takeaways

- An **API** lets our code talk to an LLM on another computer. **Request** goes out, **response** comes back.
- An **API key** is a password. Never put it in code or on GitHub. Free models may save your text.
- Paid models charge for **tokens**: input + output. Shorter prompts cost less.
- An **embedding model** turns text into numbers. Similar meaning = similar numbers.
- **Zero-shot** = no examples. **Few-shot** = show a few examples, and the model copies the pattern.
- **System prompt** = short, clear rules. **JSON** = an answer our code can use.
- **Temperature 0** = the most stable answer (like `argmax` in Day 23).

---
#  Problem: read order messages

A shop gets order messages. We want the **name, city and items** as JSON.

> *"Hi, I am Sara from Lahore. Please send 2 chai and 1 samosa."*
> → `{"name": "Sara", "city": "Lahore", "items": ["chai", "samosa"]}`

Do the **same steps** on this new job:

1. Write a short **system rule** that asks for this JSON shape.
2. Add **2 few-shot examples** (as past chat turns, like `EXAMPLES`).
3. Send the request with `temperature=0` and read `choices[0].message.content`.
4. Turn the text into a dictionary with `parse_json`.
5. Run it on 3 new messages.
6. In 2 lines: what happens if you remove the few-shot examples? Try it.